# 00 — lczerolens smoke test

**Phase 0, Chunk 1.** Verifies the local environment plumbing on CPU:

1. `lczerolens` imports cleanly.
2. A small Lc0-format network loads from HuggingFace.
3. A forward pass on the starting position produces the expected output tensors.

This is an "is my environment working?" check, **not** real analysis. The Maia-1100 net is used because it's small and downloadable; the real pinned net (BT4, per ADR-0001) lives on the remote GPU box per ADR-0002.

Notebooks per CLAUDE.md §6 are exploration-only — do not import this from production code.

In [1]:
import torch
import lczerolens
from lczerolens import LczeroBoard, LczeroModel

print(f"torch:      {torch.__version__}")
print(f"lczerolens: {lczerolens.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}  (expected: False on this laptop — see ADR-0002)")

torch:      2.12.0+cpu
lczerolens: 0.4.0
CUDA available: False  (expected: False on this laptop — see ADR-0002)


## Load Maia-1100

In [2]:
model = LczeroModel.from_hf("lczerolens/maia-1100")
model.eval()

print(f"in_keys:  {model.in_keys}")
print(f"out_keys: {model.out_keys}")

in_keys:  ['board']
out_keys: ['policy', 'wdl']


## Forward pass on the starting position

Expected output:
- **policy:** shape `(1, 1858)` — Lc0's flattened move-vector. CLAUDE.md §8 lists this 1858-element mapping as a pitfall to never change.
- **wdl:** shape `(1, 3)` — win / draw / loss probabilities.

In [3]:
board = LczeroBoard()  # starting position
print(f"FEN:                {board.fen()}")
print(f"Input tensor shape: {tuple(board.to_input_tensor().shape)}  (112 planes × 8 × 8)")

with torch.no_grad():
    out = model(board)

print()
print(f"policy: shape={tuple(out['policy'].shape)}")
print(f"wdl:    shape={tuple(out['wdl'].shape)}")

assert tuple(out['policy'].shape) == (1, 1858), "policy shape regression — should be (1, 1858)"
assert tuple(out['wdl'].shape) == (1, 3), "wdl shape regression — should be (1, 3)"
print()
print("✓ shapes match expectations")

FEN:                rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1
Input tensor shape: (112, 8, 8)  (112 planes × 8 × 8)

policy: shape=(1, 1858)
wdl:    shape=(1, 3)

✓ shapes match expectations


## Sanity check on a non-trivial position

Make sure it isn't coincidentally only working on the start position. Using Kiwipete — a standard chess testbed position with castling, en-passant, and tactical complexity.

In [4]:
kiwipete = LczeroBoard(
    "r3k2r/p1ppqpb1/bn2pnp1/3PN3/1p2P3/2N2Q1p/PPPBBPPP/R3K2R w KQkq - 0 1"
)
with torch.no_grad():
    out2 = model(kiwipete)

wdl = out2['wdl'][0]
print(f"WDL: W={wdl[0]:.3f}  D={wdl[1]:.3f}  L={wdl[2]:.3f}  (sums to {wdl.sum():.3f})")

top5 = torch.topk(out2['policy'][0], k=5)
print(f"Top-5 policy logits:  {[round(v, 3) for v in top5.values.tolist()]}")
print(f"Top-5 policy indices: {top5.indices.tolist()}")

WDL: W=0.092  D=0.050  L=0.858  (sums to 1.000)
Top-5 policy logits:  [10.147, 9.833, 9.74, 9.482, 9.361]
Top-5 policy indices: [204, 207, 378, 374, 230]


## What this verifies (and doesn't)

**Verified:** Phase 0 Chunk 1 exit criterion — *"lczerolens walkthrough notebook runs end-to-end on this machine."*

**Not verified (and shouldn't be — these are later chunks):**
- GPU forward passes (no local GPU — ADR-0002).
- BT4 inference (lives on the remote GPU box; Maia-1100 is the local stand-in).
- Lc0 binary + `--verbose-move-stats` parsing (Chunk 3).
- Stockfish (Phase 2 minimum).

**Next:** Chunk 2 (repo skeleton + initial commit), then Chunk 3 (provision cloud GPU box, mirror venv with the CUDA torch wheel).